# TaaS Studio — Browser E2E (Hypergraph Explorer + UX-24 Governance)

**Prerequisite:** first run `notebooks/colab/taas_core_studio_colab.ipynb` from top to bottom in this same runtime, including Launch Studio and Open TaaS Studio. The launcher is pinned to the exact UX-24 browser-test revision. This notebook reuses that Uvicorn process, generated `api_token`, and `proxy_url`; it does not clone another repository, start another server, bind another port, or replace the token.

The tests use the real Studio page and authenticated Core API. They exercise metadata-only demo discovery and Core hypergraph projection, then the five-view Agent Governance & Trust Center, including keyboard tab navigation, live Agent/version retrieval, and read-only assessment. Desktop/mobile screenshots are captured. No MCP tools are invoked or executed.


In [ ]:
# Reuse the running Studio launched by taas_core_studio_colab.ipynb.
# Do not start another API process or create/replace its token.
import os, pathlib
from IPython.display import display, Image

WORKDIR = pathlib.Path("/content/taas-provenance-discovery")
assert WORKDIR.is_dir(), "Run taas_core_studio_colab.ipynb first in this same Colab runtime."
assert (WORKDIR / "src" / "taas" / "studio" / "api.py").is_file()
assert "studio_process" in globals(), "The TaaS Core Studio launch cell must run first."
assert studio_process.poll() is None, "Studio is not running. Re-run its Launch Studio cell."
assert "api_token" in globals() and api_token, "The Core Studio notebook token is unavailable."
assert "proxy_url" in globals() and proxy_url, "Run the Open TaaS Studio cell first."

TAAS_STUDIO_URL = proxy_url
os.environ["TAAS_STUDIO_URL"] = TAAS_STUDIO_URL
os.environ["TAAS_STUDIO_API_TOKEN"] = api_token
os.environ["TAAS_E2E_ARTIFACT_DIR"] = "/content/taas-e2e-artifacts"
print("Reusing Studio process PID:", studio_process.pid)
print("Existing proxy URL:", TAAS_STUDIO_URL)
print("Existing API token: present (value hidden)")
%cd /content/taas-provenance-discovery



In [ ]:
!pip -q install playwright
!playwright install --with-deps chromium
!python -m compileall -q src/taas tests/test_studio_browser_e2e.py tests/test_studio_ux24_governance.py


In [ ]:
!pytest -q tests/test_retrieval_degenerate.py tests/test_studio_hypergraph_explorer.py tests/test_studio_navigation_shell.py tests/test_studio_professional_ux.py tests/test_hypergraph_projection.py tests/test_engine_certificate_gate.py tests/test_studio_ux24_governance.py tests/test_studio_browser_e2e.py -s


In [ ]:
from pathlib import Path
from IPython.display import display, Image

artifact_dir = Path(os.environ.get('TAAS_E2E_ARTIFACT_DIR', '/content/taas-e2e-artifacts'))
artifacts = sorted([*artifact_dir.glob('hypergraph-explorer-*.png'), *artifact_dir.glob('ux24-governance-*.png')]) if artifact_dir.exists() else []

if not artifacts:
    print('No browser screenshots were exported.')
    print('Expected directory:', artifact_dir)
    print('Check the pytest cell above:')
    print('- If the browser tests FAILED, fix the first pytest assertion/error before expecting screenshots.')
    print('- If the browser tests were SKIPPED, verify TAAS_STUDIO_URL and TAAS_STUDIO_API_TOKEN were set in the same runtime.')
    print('- If pytest PASSED but files are absent, rerun the browser test cell and inspect its output.')
    raise RuntimeError('Browser screenshot artifacts are unavailable; see the pytest result above.')

for artifact in artifacts:
    assert artifact.is_file() and artifact.stat().st_size > 0, f'Invalid screenshot artifact: {artifact}'
    print(f'{artifact.name} — {artifact.stat().st_size:,} bytes')
    display(Image(filename=str(artifact)))


## Runtime reuse and validation boundary

- Reuses the exact server process, token and proxy URL created by `taas_core_studio_colab.ipynb`.
- The launcher checks out the pinned UX-24 code/test revision before starting Studio.
- Does not reinstall TaaS, create a second token, launch another Uvicorn process or call `proxyPort`.
- Chromium authenticates through the existing notebook token and Colab proxy.
- Hypergraph discovery must return `taas.hypergraph-projection/v1` and `execution_performed=false`.
- UX-24 retrieves Agent/version, release-readiness, policy verdict, approval records, lifecycle events and audit verification through existing read endpoints only.
- Exercises keyboard navigation across all five UX-24 tabs and both desktop/mobile viewports.
- Rejects UX-24 write requests and MCP inspection/invocation requests; configuration listing is not MCP server contact.

A pass validates these browser journeys against the exact pinned Colab process only. It does not prove production identity, live MCP execution, authorization, cross-browser compatibility, screen-reader certification or deployment resilience. Keep Colab output distinct from GitHub Actions status.
